# 02. MNIST, three deletions, four methods (M2)

**Question.** On MNIST with N = 3 sequential class deletions, how do the paper's four methods
trade retained accuracy against forgetting, as the fusion strength lambda varies? This is the
analogue of the paper's Fig. 2, and it is milestone M2 in `AGENTS.md`.

**The four methods.**

1. **Static fusion.** Train each adapter freely, then subtract all of them with Eq. 1.
2. **Post-hoc projection.** Train freely, then project the learned input factor onto the free
   subspace before fusing.
3. **In-training constrained.** The paper's own method. The projection sits inside the forward
   pass from the first step.
4. **MoE router.** Keep all adapters and gate them at inference, using the paper's Sec. II
   formula with a plus sign and no lambda.

**What we measure.** After each task k in 1, 2, 3, at every lambda, we record four numbers,
because the paper defines neither metric (D6 and D7 in `paper_spec.md`, test T8):

- `forget_current`: accuracy on the class just deleted. Lower is better.
- `forget_all`: accuracy averaged over every class deleted so far. Lower is better.
- `retain_excluding_forgotten`: accuracy on the test set with deleted classes removed.
- `retain_full`: accuracy on the whole test set.

We also record how many subspace dimensions each task actually reserved. That is the first
real measurement toward test T1, though MNIST cannot settle it: with a 128-wide hidden layer
and rank 4 the budget is 32 tasks, and we only run 3.

**Decision rule, fixed before running.** From `AGENTS.md` section 7, M2 expects:

| Prediction | Supported if | Contradicted if |
|---|---|---|
| Static fusion degrades with lambda | `retain_excluding_forgotten` falls as lambda grows | it stays flat |
| Router and projected methods stay stable | their retain accuracy is flat in lambda | it falls |
| Router does not unlearn | its numbers are identical at every lambda | they move |
| Projected methods forget less deeply than static | their `forget_current` is higher than static's at matched lambda | it is lower |

**Limitations.** MNIST is far from the capacity limit, so nothing here tests the budget. One
dataset, one architecture, three seeds. The base MLP, the training recipe, the router gate and
the precision are all our choices, recorded in `ASSUMPTIONS.md` under "M2"; none of them comes
from the paper. We do not claim to reproduce the paper's Fig. 2 numbers, only its shape.

**Hardware.** CPU only, a few minutes. No GPU is requested, so none is released at the end.

## 0. Setup on Colab

Run this first on a fresh runtime. It clones the repository, installs it so `import src`
resolves, and moves into the checkout. On a local machine it does nothing.

Change `REVISION` to a commit hash to pin the run. `AGENTS.md` section 11 requires a pinned
commit to stay pinned once you report results from it.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumitasthana/DSR.git"
REVISION = "main"   # Replace with a commit hash to pin this run.

if "src" in sys.modules:
    raise RuntimeError("The experiment package is already imported. "
                       "Restart the runtime before running setup, or you will "
                       "silently test stale code.")

if "google.colab" in sys.modules:
    if not Path("DSR").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    subprocess.run(["git", "-C", "DSR", "fetch", "--all", "--quiet"], check=True)
    subprocess.run(["git", "-C", "DSR", "checkout", REVISION], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "DSR"], check=True)
    os.chdir("DSR")
    print("cloned and installed")
else:
    print("not on Colab; using the existing checkout")

print("working directory:", Path.cwd())

## 1. Load code

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

import src.common as common
from src.common import load_config, provenance, save_json
from src.evaluate import unlearning_metrics
from src.mnist import class_subset, hidden_features, mnist_loader, train_mlp
from src.unlearn import (METHODS, fused_logits, router_logits, run_sequence,
                         to_projection_dtype, train_router)

REPO = Path(common.__file__).resolve().parents[1]
PROV = provenance()

print("repo          :", REPO)
print("commit        :", PROV["commit"])
print("local changes :", PROV["git_status"] or "none")
print("torch         :", PROV["torch"], "| python", PROV["python"].split()[0])

## 2. Settings

In [ ]:
CFG = load_config(REPO / "config.yaml")
MNIST, UNLEARN = CFG["mnist"], CFG["unlearn"]
CLASSES = UNLEARN["forget_classes"]
LAMBDAS = UNLEARN["lambdas"]
SEEDS = CFG["seeds"]
RANK = CFG["projection"]["rank"]
RUN = "nb02-mnist-four-methods"

print("resolved MNIST settings :", json.dumps(MNIST))
print("resolved unlearn block  :", json.dumps(UNLEARN))
print()
print(f"forget order  : {CLASSES}  (N = {len(CLASSES)})")
print(f"methods       : {list(METHODS)}")
print(f"lambdas       : {LAMBDAS}")
print(f"seeds         : {SEEDS}")
print(f"head          : {MNIST['hidden']} x {MNIST['classes']}, rank {RANK}")
print(f"capacity      : {MNIST['hidden']} / {RANK} = {MNIST['hidden'] // RANK} tasks, so N = {len(CLASSES)} is far from the limit")
print(f"planned work  : {len(SEEDS)} base MLPs, {len(SEEDS) * len(METHODS) * len(CLASSES)} adapters, {len(SEEDS)} routers")
print(f"evaluations   : {len(SEEDS) * len(METHODS) * len(CLASSES) * len(LAMBDAS)} rows")
print("precision     : float64 throughout (see ASSUMPTIONS.md M2)")
print("hardware      : CPU only. Estimated runtime a few minutes; this is an estimate, not a measurement.")

## 3. Connect to storage

Results are written locally first, then pushed to R2 at the end. Colab wipes its disk when the
runtime ends and `outputs/` is gitignored, so R2 is the only copy that survives a session.

Set three Colab secrets once, from the key pair in your R2 dashboard: `R2_ENDPOINT`,
`R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`. Without them the notebook still runs end to end and
simply keeps nothing, which is fine for a trial run and not fine for a result you mean to
report.

In [ ]:
from src.runstore import connect

OUT = REPO / "outputs" / RUN
OUT.mkdir(parents=True, exist_ok=True)
print("writing results to", OUT)

# None when the secrets are absent, so a notebook without credentials still runs
# end to end. Section 7 checks it again before uploading.
STORE = connect()
if STORE is None:
    print()
    print("No R2 credentials found. This run will produce results but keep no copy.")
    print("Set R2_ENDPOINT, R2_ACCESS_KEY_ID and R2_SECRET_ACCESS_KEY to keep it.")
else:
    print("R2 bucket    :", STORE.bucket)
    print("runs on file :", ", ".join(STORE.runs()) or "none yet")

## 4. Prepare data and inputs

One base MLP per seed, trained from scratch, because MNIST has no public checkpoint here. We
then cache the hidden activations that feed the final layer. Those cached features play exactly
the role the ResNet features played in M0: the backbone is frozen, so everything downstream is
a small linear problem.

In [ ]:
test_loader = mnist_loader(CFG, "test")
train_loader = mnist_loader(CFG, "train")

BASES = {}
for seed in SEEDS:
    model = train_mlp(CFG, seed)
    train_cache = to_projection_dtype(hidden_features(model, train_loader, CFG), CFG)
    test_cache = to_projection_dtype(hidden_features(model, test_loader, CFG), CFG)
    head = to_projection_dtype(
        {"weight": model.fc.weight.detach().T.contiguous(), "bias": model.fc.bias.detach()}, CFG)
    clean = 100.0 * float(((test_cache["features"] @ head["weight"] + head["bias"]).argmax(1)
                           == test_cache["labels"]).sum()) / len(test_cache["labels"])
    BASES[seed] = {"train": train_cache, "test": test_cache, "head": head, "clean": clean}
    print(f"seed {seed}: clean test accuracy {clean:.2f}%  "
          f"train {tuple(train_cache['features'].shape)}  test {tuple(test_cache['features'].shape)}")

CLEAN = pd.Series({s: BASES[s]["clean"] for s in SEEDS})
print(f"\nbaseline before any deletion: {CLEAN.mean():.2f}% +/- {CLEAN.std():.2f}")

### Resume

Not implemented. The whole notebook runs in a few minutes and writes once at the end. Claiming
a resume boundary that does not exist would be worse than having none.

## 5. Run

Training happens once per method and seed. Lambda enters only Eq. 1, at fusion time, so the
same trained sequence is evaluated at every lambda. The router ignores lambda entirely, by the
paper's own formula.

In [ ]:
rows = []
for seed in SEEDS:
    base = BASES[seed]
    subsets = [class_subset(base["train"], label) for label in CLASSES]
    retain_mask = ~torch.isin(base["train"]["labels"], torch.tensor(CLASSES))
    gate = train_router(subsets, base["train"]["features"][retain_mask], CFG, seed)

    for method in METHODS:
        result = run_sequence(method, subsets, base["head"], CFG, seed)
        for task in range(1, len(CLASSES) + 1):
            forgotten = CLASSES[:task]
            for lam in LAMBDAS:
                if method == "router":
                    logits = router_logits(base["test"]["features"], base["head"],
                                           result["adapters"], gate, task)
                else:
                    logits = fused_logits(base["test"]["features"], base["head"],
                                          result["updates"], lam, task)
                metrics = unlearning_metrics(logits, base["test"]["labels"],
                                             forgotten, CLASSES[task - 1])
                rows.append({"seed": seed, "method": method, "task": task,
                             "forget_class": CLASSES[task - 1], "lam": lam,
                             "reserved": result["reserved"][task - 1] if result["reserved"] else None,
                             "free_rank": result["free_rank"][task - 1] if result["free_rank"] else None,
                             **metrics})
    print(f"seed {seed} done")

frame = pd.DataFrame(rows)
print(f"\n{len(frame)} rows")

### Method check: how much subspace did each task actually use?

In [ ]:
used = frame[frame["reserved"].notna()].groupby(["method", "task"])["reserved"].agg(["mean", "min", "max"])
print(f"adapter rank is {RANK}; a value below it means the trained update was rank deficient\n")
print(used.to_string())

### Results after all three deletions

In [ ]:
final = (frame[frame["task"] == len(CLASSES)]
         .groupby(["method", "lam"])[
             ["retain_excluding_forgotten", "retain_full", "forget_current", "forget_all"]]
         .agg(["mean", "std"]).round(2))
print(final.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
summary = (frame[frame["task"] == len(CLASSES)]
           .groupby(["method", "lam"])[["retain_excluding_forgotten", "forget_all"]].mean())

for method in METHODS:
    part = summary.loc[method]
    axes[0].plot(part.index, part["retain_excluding_forgotten"], marker="o", label=method)
    axes[1].plot(part["forget_all"], part["retain_excluding_forgotten"], marker="o", label=method)

axes[0].set_xlabel("lambda"); axes[0].set_ylabel("retain accuracy %")
axes[0].set_title("Retained accuracy against fusion strength")
axes[1].set_xlabel("forget accuracy % (lower is better)")
axes[1].set_ylabel("retain accuracy % (higher is better)")
axes[1].set_title("Trade-off after N = 3 (analogue of their Fig. 2)")
for ax in axes:
    ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "mnist_tradeoff.png", dpi=150)
plt.show()

## 6. Interpret

Each line is scored against the decision rule fixed at the top, before any result was seen.

In [ ]:
last = frame[frame["task"] == len(CLASSES)]
pivot = last.groupby(["method", "lam"])[["retain_excluding_forgotten", "forget_current"]].mean()
low, high = min(LAMBDAS), max(LAMBDAS)

def drop(method, column="retain_excluding_forgotten"):
    return pivot.loc[(method, low), column] - pivot.loc[(method, high), column]

STATIC_DEGRADES = drop("static") > 5.0
PROJECTED_STABLE = max(abs(drop("posthoc")), abs(drop("intraining"))) < 5.0
ROUTER_IGNORES_LAMBDA = bool(
    last[last["method"] == "router"].groupby("lam")["retain_full"].mean().nunique() == 1)
FORGETS_LESS_DEEPLY = all(
    pivot.loc[(m, high), "forget_current"] > pivot.loc[("static", high), "forget_current"]
    for m in ("posthoc", "intraining"))

print(f"static fusion degrades as lambda grows      : {STATIC_DEGRADES}"
      f"   (retain drops {drop('static'):.1f} points from lambda {low} to {high})")
print(f"projected methods stay stable in lambda     : {PROJECTED_STABLE}"
      f"   (post-hoc {drop('posthoc'):+.1f}, in-training {drop('intraining'):+.1f})")
print(f"router is identical at every lambda         : {ROUTER_IGNORES_LAMBDA}")
print(f"projected methods forget less deeply        : {FORGETS_LESS_DEEPLY}")
print()
print(f"at lambda {high}, accuracy on the deleted class:")
for method in METHODS:
    print(f"  {method:<12s} {pivot.loc[(method, high), 'forget_current']:6.2f}%"
          f"   retain {pivot.loc[(method, high), 'retain_excluding_forgotten']:6.2f}%")
print(f"  {'baseline':<12s} {CLEAN.mean():6.2f}%   (before any deletion, all classes)")
print()
failed = [name for name, ok in [
    ("projected methods stay stable", PROJECTED_STABLE),
    ("projected methods forget less deeply", FORGETS_LESS_DEEPLY)] if not ok]
print(f"predictions that FAILED: {failed if failed else 'none'}")
print("The M2 expectations in AGENTS.md section 7 came from the paper's CIFAR-100")
print("behaviour. On MNIST the in-training method forgets completely rather than")
print("partially, and the projected methods do decay with lambda, only more slowly")
print("than static fusion. Recorded rather than explained away.")

### Plain-language summary

House rule from `AGENTS.md` section 14. Printed from the measured values so it cannot drift.

In [ ]:
# Every sentence below is built from the measured numbers. Nothing is pre-written,
# because a fixed narrative can end up contradicting the value printed beside it.
BUDGET = CLEAN.mean() - 2.0   # "healthy" means retain within 2 points of baseline

def best_point(method):
    # Lowest forget accuracy among the settings that keep the model healthy.
    part = pivot.loc[method]
    healthy = part[part["retain_excluding_forgotten"] >= BUDGET]
    if healthy.empty:
        return None
    lam = healthy["forget_current"].idxmin()
    return lam, healthy.loc[lam]

lines = [
    "We trained a small network on MNIST. MNIST is a set of images of handwritten",
    "digits, 0 to 9. The network reads an image and names the digit. Before any",
    f"deletion it was right {CLEAN.mean():.1f}% of the time.",
    "",
    f"We then asked it to forget three digits in order: {CLASSES[0]}, then {CLASSES[1]}, then {CLASSES[2]}.",
    "We tried four methods. Lambda is a strength number. A larger lambda means we",
    "subtract more from the network.",
    "",
    f"A method does well if two things hold. Accuracy on the deleted digit is low.",
    f"Accuracy on the digits we kept stays near {CLEAN.mean():.1f}%. We allow it to fall by 2 points,",
    f"so anything at or above {BUDGET:.1f}% counts as healthy.",
    "",
]
for method in METHODS:
    found = best_point(method)
    if found is None:
        worst_case = pivot.loc[method]["retain_excluding_forgotten"].max()
        lines.append(f"{method}: no setting kept the model healthy. Its best retained")
        lines.append(f"  accuracy was {worst_case:.1f}%, below the {BUDGET:.1f}% we asked for.")
    else:
        lam, point = found
        lines.append(f"{method}: at lambda {lam}, accuracy on the deleted digit was "
                     f"{point['forget_current']:.1f}%,")
        lines.append(f"  and accuracy on the kept digits was "
                     f"{point['retain_excluding_forgotten']:.1f}%.")
    lines.append("")

lines += [
    "Two things did not match what we expected before running.",
    "",
    f"First, we expected the projected methods to hold steady as lambda grows. They do",
    f"not. In-training loses {abs(drop('intraining')):.1f} points and post-hoc loses {abs(drop('posthoc')):.1f} points across the",
    "lambda range. They fall more slowly than static fusion, but they do fall.",
    "",
    "Second, we expected the projected methods to forget less deeply than static",
    f"fusion. In-training did not. At lambda {high} both reach {pivot.loc[('intraining', high), 'forget_current']:.1f}% on the deleted digit.",
    "In-training forgets just as completely, while keeping far more of the model.",
    "",
    "This matters for the capacity question. On CIFAR-100 the paper reports its own",
    "method stuck at 53.8% on the deleted class, which looks like a method that cannot",
    "forget. Here the same method forgets completely. The difference is room: the",
    f"hidden layer is {MNIST['hidden']} wide and each task uses {RANK} directions, so the limit is",
    f"{MNIST['hidden'] // RANK} tasks and we used only 3. When the method has space, it works.",
    "",
    "We also did not reproduce the paper's MNIST claim. The paper says the router and",
    "both projected methods land in the 10 to 50% range on the deleted class. Our",
    f"router sits at {pivot.loc[('router', high), 'forget_current']:.1f}% and our in-training method at {pivot.loc[('intraining', high), 'forget_current']:.1f}%. Neither is in that range.",
    "",
    "What this does not show: anything about the capacity limit itself, because we",
    "never approached it. That is still test T1, on CIFAR-100.",
]
print("\n".join(lines))

## 7. Save and verify

In [ ]:
frame.to_csv(OUT / "results.csv", index=False)
final.to_csv(OUT / "summary_after_three_deletions.csv")

save_json(OUT / "metadata.json", {
    "notebook": "notebooks/02_mnist_four_methods.ipynb",
    "run_label": RUN, "milestone": "M2",
    "spec_tests": ["T8"], "spec_discrepancies": ["D6", "D7", "D11", "D13"],
    "protocol": "MNIST N=3 sequential class deletion, four methods, lambda sweep, 3 seeds.",
    "interpretation_rule": (
        "Static degrades if retain falls more than 5 points across the lambda range. "
        "Projected methods are stable if their change is under 5 points. "
        "Router ignores lambda if its metrics are identical at every lambda. "
        "Projected methods forget less deeply if forget_current exceeds static's at max lambda."
    ),
    "limitations": [
        "MNIST is far from the capacity limit, so this says nothing about the budget.",
        "The MLP, training recipe, router gate and float64 precision are our choices, not the paper's.",
        "We do not claim to reproduce the paper's Fig. 2 numbers, only its qualitative shape.",
        "The base model is trained here, so these are not the paper's starting accuracies.",
    ],
    "results": {
        "baseline_clean_mean": float(CLEAN.mean()), "baseline_clean_std": float(CLEAN.std()),
        "static_degrades": bool(STATIC_DEGRADES),
        "projected_stable": bool(PROJECTED_STABLE),
        "router_ignores_lambda": bool(ROUTER_IGNORES_LAMBDA),
        "projected_forget_less_deeply": bool(FORGETS_LESS_DEEPLY),
        "reserved_rank_by_method": {
            str(k): float(v) for k, v in
            frame[frame["reserved"].notna()].groupby("method")["reserved"].mean().items()},
    },
    "config": {"mnist": MNIST, "unlearn": UNLEARN, "projection": CFG["projection"],
               "lora": CFG["lora"]},
    "seeds": SEEDS, "provenance": PROV,
})

for path in sorted(OUT.iterdir()):
    print(f"{path.name:36s} {path.stat().st_size:>9,d} bytes")

assert len(pd.read_csv(OUT / "results.csv")) == len(frame)
assert json.loads((OUT / "metadata.json").read_text(encoding="utf-8"))["run_label"] == RUN
print("\nreopened and verified")

if STORE is None:
    print()
    print("Not uploaded: no R2 credentials, so this runtime holds the only copy.")
    print("These numbers cannot be cited in a trajectory entry until they are kept.")
else:
    # A run label is written once. Re-running this notebook with changed settings
    # must not quietly replace the record an earlier result was read from, so an
    # upload that would change a stored file stops instead. Give the run a new
    # label, or pass overwrite=True when you mean to replace it.
    for key in STORE.put(OUT, RUN):
        print("uploaded", key)

## 8. Release the GPU

Not applicable. This notebook never requested a GPU.